# M04 · Embeddings e pesquisa semântica: prática

Neste notebook vais:

1. Gerar embeddings e explorar similaridades.
2. Montar um conjunto de avaliação com notícias em português (título como pergunta, artigo como resposta).
3. Comparar BM25, pesquisa vetorial exata, HNSW e pesquisa híbrida com recall@k e MRR.
4. Usar o Qdrant com filtros de metadados.

**Requisitos:** `uv sync --extra rag`. Tudo corre localmente em CPU (a codificação de 5 000 artigos demora alguns minutos; com GPU segundos).

Lição de apoio: [licao.html](licao.html)

In [ ]:
import time, re
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

MODELO_EMB = "intfloat/multilingual-e5-base"
emb = SentenceTransformer(MODELO_EMB)
print(emb, emb.get_sentence_embedding_dimension(), "dimensões")

## 1. Primeiros embeddings

In [ ]:
frases = [
    "Como devolvo uma encomenda?",
    "Tem 30 dias para devolver qualquer artigo sem justificação.",
    "Reembolsamos o valor no mesmo meio de pagamento.",
    "As entregas nos Açores demoram 3 a 5 dias úteis.",
    "O Benfica venceu o Sporting por 2-1.",
]
v = emb.encode(["query: " + frases[0]] + ["passage: " + f for f in frases[1:]], normalize_embeddings=True)
pd.Series(v[0] @ v[1:].T, index=frases[1:]).sort_values(ascending=False).round(3)

**Exercício 1.1.** Repete sem os prefixos `query:` e `passage:`. As pontuações mudam? A ordem mantém-se?

**Exercício 1.2.** Calcula a matriz de similaridade entre as 5 frases e mostra-a como heatmap (`plt.imshow`).

## 2. Conjunto de avaliação

Usamos a parte portuguesa do XL-Sum (notícias da BBC Brasil com título e texto). Cada título é uma pergunta e o artigo correspondente é o único documento relevante. É uma tarefa difícil e realista: o título raramente repete as palavras do texto.

In [ ]:
from datasets import load_dataset

ds = load_dataset("csebuetnlp/xlsum", "portuguese", split="test")
N = 5000
df = ds.select(range(min(N, len(ds)))).to_pandas()[["id", "title", "text"]]
df["text"] = df.text.str.slice(0, 2000)  # os modelos têm limite de tokens; os primeiros parágrafos chegam
print(len(df), "artigos")
df.head(3)

In [ ]:
perguntas = df.title.tolist()
docs = df.text.tolist()
relevante = np.arange(len(df))    # a pergunta i tem como resposta o documento i

def avaliar(resultados, ks=(1, 5, 10)):
    """resultados: array (n_perguntas, top) com índices de documentos."""
    r = {f"recall@{k}": np.mean([relevante[i] in resultados[i][:k] for i in range(len(resultados))]) for k in ks}
    rr = [1 / (list(res).index(relevante[i]) + 1) if relevante[i] in res else 0 for i, res in enumerate(resultados)]
    r["MRR@10"] = np.mean(rr)
    return {k: round(v, 3) for k, v in r.items()}

## 3. BM25

In [ ]:
from rank_bm25 import BM25Okapi

def tokenizar(t):
    return re.findall(r"\w+", t.lower())

t0 = time.time()
bm25 = BM25Okapi([tokenizar(d) for d in docs])
res_bm25 = np.array([np.argsort(-bm25.get_scores(tokenizar(q)))[:50] for q in perguntas[:1000]])
print(f"BM25 em {time.time() - t0:.0f}s")
resultados = {"BM25": avaliar(res_bm25)}
resultados

Avaliamos as primeiras 1000 perguntas para o BM25 não demorar demasiado (o `rank_bm25` é simples e lento). Os outros métodos usam as mesmas 1000.

## 4. Pesquisa vetorial exata

In [ ]:
t0 = time.time()
D = emb.encode(["passage: " + d for d in docs], normalize_embeddings=True, batch_size=32, show_progress_bar=True)
Q = emb.encode(["query: " + q for q in perguntas[:1000]], normalize_embeddings=True, batch_size=64)
print(f"codificação em {time.time() - t0:.0f}s  |  D {D.shape}")

t0 = time.time()
res_vet = np.argsort(-(Q @ D.T), axis=1)[:, :50]
print(f"pesquisa exata: {(time.time() - t0) * 1000:.0f} ms para 1000 perguntas")
resultados["Vetorial (exata)"] = avaliar(res_vet)
pd.DataFrame(resultados).T

## 5. FAISS: exata vs HNSW

In [ ]:
import faiss

d = D.shape[1]
flat = faiss.IndexFlatIP(d); flat.add(D)
hnsw = faiss.IndexHNSWFlat(d, 32, faiss.METRIC_INNER_PRODUCT)
hnsw.hnsw.efConstruction = 200
t0 = time.time(); hnsw.add(D); print(f"HNSW construído em {time.time() - t0:.1f}s")

for ef in (16, 64, 256):
    hnsw.hnsw.efSearch = ef
    t0 = time.time(); _, I = hnsw.search(Q, 50); ms = (time.time() - t0) * 1000
    print(f"efSearch={ef:3d}  {ms:6.0f} ms  ", avaliar(I))

Com 5 000 documentos a pesquisa exata já é instantânea; o HNSW compensa a partir de centenas de milhares. Repara como `efSearch` troca velocidade por recall.

**Exercício 5.1.** Replica os documentos 100 vezes com ruído (`D + 0.01 * np.random.randn(...)`, renormalizados) para chegares a 500 mil vetores e compara outra vez os tempos da pesquisa exata e do HNSW.

## 6. Pesquisa híbrida com RRF

In [ ]:
def rrf(listas, k=60, top=50):
    pont = {}
    for lista in listas:
        for pos, doc in enumerate(lista):
            pont[doc] = pont.get(doc, 0) + 1 / (k + pos + 1)
    return sorted(pont, key=pont.get, reverse=True)[:top]

res_hib = np.array([rrf([res_bm25[i], res_vet[i]]) for i in range(len(res_vet))])
resultados["Híbrida (RRF)"] = avaliar(res_hib)
pd.DataFrame(resultados).T

**Exercício 6.1.** Encontra 5 perguntas em que o BM25 acerta no top 1 e a pesquisa vetorial falha, e 5 ao contrário. O que têm em comum?

In [ ]:
bm_ok = res_bm25[:, 0] == relevante[:1000]
vet_ok = res_vet[:, 0] == relevante[:1000]
print("Só BM25 acerta:")
for i in np.where(bm_ok & ~vet_ok)[0][:5]:
    print("  ", perguntas[i])
print("Só vetorial acerta:")
for i in np.where(~bm_ok & vet_ok)[0][:5]:
    print("  ", perguntas[i])

## 7. Qdrant com metadados

O Qdrant tem um modo local (sem servidor) ótimo para desenvolvimento. Guardamos o vetor e metadados (aqui, o comprimento do artigo e uma categoria fictícia) e filtramos na pesquisa.

In [ ]:
from qdrant_client import QdrantClient, models

qd = QdrantClient(":memory:")    # ou QdrantClient(path="qdrant_db") para persistir no disco
qd.create_collection("noticias", vectors_config=models.VectorParams(size=d, distance=models.Distance.COSINE))

rng = np.random.default_rng(0)
categorias = rng.choice(["política", "economia", "ciência", "desporto"], len(df))
qd.upload_points("noticias", [
    models.PointStruct(id=i, vector=D[i].tolist(), payload={"titulo": df.title[i], "categoria": categorias[i], "n_chars": len(docs[i])})
    for i in range(len(df))
])

q = emb.encode("query: inflação e taxas de juro no Brasil", normalize_embeddings=True)
hits = qd.query_points("noticias", query=q.tolist(), limit=5,
                       query_filter=models.Filter(must=[models.FieldCondition(key="categoria", match=models.MatchValue(value="economia"))])).points
for h in hits:
    print(f"{h.score:.3f}  [{h.payload['categoria']}]  {h.payload['titulo']}")

## Use case: motor de pesquisa semântica de notícias

Em `use-case/` tens o motor completo, reutilizável nos módulos seguintes:

- `motor.py`: classe `Motor` com indexação em Qdrant (persistente), BM25 e pesquisa vetorial e híbrida.
- `avaliar.py`: corre o conjunto de avaliação e escreve uma tabela comparativa.

```bash
cd modulos/04-embeddings-e-pesquisa-semantica/use-case
uv run python avaliar.py --n 5000 --perguntas 1000
```

**Desafio:** compara três modelos de embeddings (`multilingual-e5-base`, `bge-m3` e um modelo via API) em recall@5, tempo de codificação e custo. Escreve as conclusões no README do use case.